# Evolve: Classify with Gold Set

This notebook:
1. Parses the SME-authored codebook (`data/Evolve gold set.xlsx`, sheet `Definition`) into a 2-level taxonomy (L1 = Contact Reason, L2 = Root Cause) and writes it into the `taxonomy` sheet of `definitions/evolve_pipeline_definition.xlsx`.
2. Disables the old, separate `root_cause` module (kept, not deleted) and migrates its justification/evidence fields into a single hierarchical `contact_reason` module.
3. Builds the pipeline (`rca build`) and converts the 284-ticket gold set (`data/Evolve gold set.xlsx`, sheet `evolve`) into few-shot examples via `FeedbackStore`.
4. Classifies the 2026 slice of `data/Evolve_Collapsed.xlsx` (126,504 rows total; 20,070 dated in 2026 -- the in-scope slice) with the LLM pipeline.
5. Evaluates accuracy against the gold set via 5-fold cross-validation.

**Deviation from the `04_leapspace_classify_with_gold_set.ipynb` precedent:** LeapSpace had no pre-existing taxonomy, so its companion `scripts/build_leapspace_definition.py` hand-coded the Contact-Reason/Root-Cause hierarchy. Evolve already has a real, structured SME codebook (the `Definition` sheet), so this notebook parses it mechanically -- as Sections 1-2 below -- rather than using a separate script, since the ask was specifically for a notebook that performs the update.

**Update -- LLM-checked corrections (`data/evolve_Root_Cause_errors.xlsx`).** Section 4 now overlays a round of LLM-checked label corrections on top of the gold set. That file reviews the 170 tickets whose Root-Cause prediction disagreed with gold in a prior run; its `Final cr` / `Final rc` columns hold the reviewer's verdicts (51 filled) and `Reason` explains each. Corrections are merged by `Ticket Number` (24 CR + 32 RC labels change); rows left blank keep their original gold label. A few loosely-written Contact Reason names are canonicalized to the taxonomy labels. Each reviewer `Reason` is wired into the few-shot `reason` field (the highest-weight signal in the `smart` strategy). The taxonomy is unchanged.

## 0. Configuration

In [ ]:
import os
from pathlib import Path as _P
_root = _P.cwd()
while not (_root / "definitions").is_dir() and _root.parent != _root:
    _root = _root.parent
os.chdir(_root)  # the use-case folder

from pathlib import Path
BASE = Path().resolve()

# -- Paths --------------------------------------------------------------
GOLD_SET_PATH   = BASE / "data/Evolve gold set.xlsx"
CHECKED_SET_PATH = BASE / "data/evolve_Root_Cause_errors.xlsx"
DEFINITION_XLSX = BASE / "definitions/evolve_pipeline_definition.xlsx"
RAW_DATA_PATH   = BASE / "data/Evolve_Collapsed.xlsx"
OUTPUT_PATH     = BASE / "runs/output/evolve_enriched.xlsx"
FEEDBACK_DIR    = BASE / "feedback/evolve"
TAXONOMY_DIR    = BASE / "data/evolve_taxonomies"      # product-specific, avoids clobbering generated/taxonomies/
CODE_MAP_PATH   = BASE / "data/evolve_code_map.json"

# -- Scope: 2026 only -----------------------------------------------------
# Evolve_Collapsed.xlsx spans 2023-10-30 to 2026-02-10 (2 rows in 2023, 139 in
# 2024, 106,293 in 2025, 20,070 in 2026). Classification targets only the
# 2026 slice -- 2023-2025 rows are out of scope for this notebook.
SCOPE_YEAR = 2026

# -- Sampling --------------------------------------------------------------
# The full 2026 slice is 20,070 rows. Real LLM API cost is involved
# (~$0.006-0.007/ticket measured on both LeapSpace and this notebook's own
# smoke test). Default to a small sample; set SAMPLE_N = None for the full
# 2026 run once you are happy with sample accuracy.
SAMPLE_N    = None      # <-- change to None for the full 20,070-row 2026 run
SAMPLE_SEED = 42

# -- Classification settings ------------------------------------------------
TEXT_COL          = "Conversation"
ID_COL            = "Reference #"
MODULES           = ["contact_reason"]
FEEDBACK_STRATEGY = "smart"
FEEDBACK_MAX      = 5
BATCH_SIZE        = 10
CONCURRENCY       = 4

# -- Subprocess environment ------------------------------------------------
# rca build / rca run print Unicode box-drawing separators.
# On Windows, a piped (non-console) stdout defaults to the system codepage
# (cp1252) unless UTF-8 is forced, which crashes those prints. Force it for
# every subprocess call below.
import os
_SUBPROCESS_ENV = os.environ.copy()
_SUBPROCESS_ENV["PYTHONUTF8"] = "1"
_SUBPROCESS_ENV["PYTHONIOENCODING"] = "utf-8"

# -- Sanity check ------------------------------------------------------------
assert GOLD_SET_PATH.exists(), f"Gold set not found: {GOLD_SET_PATH}"
assert CHECKED_SET_PATH.exists(), f"Checked/errors set not found: {CHECKED_SET_PATH}"
assert RAW_DATA_PATH.exists(), f"Raw data not found: {RAW_DATA_PATH}"
assert DEFINITION_XLSX.exists(), f"Definition file not found: {DEFINITION_XLSX}"
print(f"Paths OK  |  SCOPE_YEAR: {SCOPE_YEAR}  |  SAMPLE_N: {SAMPLE_N}")

## 1. Build the taxonomy from the `Definition` sheet

The `Definition` sheet (65 rows) is already shaped like a taxonomy: each `"Contact reason"` row starts a new L1 group, and the `"Root cause"` rows immediately following it are that group's L2 children. This is parsed mechanically -- no hand-authored mapping needed.

In [ ]:
import pandas as pd

def parse_definition_sheet(defn_df: pd.DataFrame) -> list[dict]:
    """defn_df: Definition sheet read with header=None (3 unnamed cols: marker, label, description)."""
    groups = []
    current = None
    for _, row in defn_df.iterrows():
        marker = str(row[0]).strip()
        label  = str(row[1]).strip()
        desc   = str(row[2]).strip() if pd.notna(row[2]) else ""
        if marker == "Contact reason":
            current = {"cr_label": label, "cr_description": desc, "children": []}
            groups.append(current)
        elif marker == "Root cause":
            if current is None:
                raise ValueError(f"Root cause row {label!r} has no preceding Contact reason row")
            current["children"].append({"rc_label": label, "rc_description": desc})
        else:
            raise ValueError(f"Unrecognised marker column value: {marker!r}")
    return groups


def build_taxonomy_rows(groups: list[dict]) -> pd.DataFrame:
    rows = []
    for g in groups:
        rows.append({
            "module_name": "contact_reason", "level": "L1",
            "label": g["cr_label"], "description": g["cr_description"], "notes": "",
        })
        for child in g["children"]:
            rows.append({
                "module_name": "contact_reason", "level": "L2",
                "label": child["rc_label"], "description": child["rc_description"], "notes": "",
            })
    return pd.DataFrame(rows)


defn_raw = pd.read_excel(GOLD_SET_PATH, sheet_name="Definition", header=None)
groups = parse_definition_sheet(defn_raw)
print(f"Parsed {len(groups)} Contact Reason groups, {sum(len(g['children']) for g in groups)} Root Cause leaves")
for g in groups:
    print(f"  {g['cr_label']!r}  ({len(g['children'])} RC)")

In [ ]:
taxonomy_df = build_taxonomy_rows(groups)
print(f"Taxonomy sheet: {len(taxonomy_df)} rows "
      f"({(taxonomy_df['level']=='L1').sum()} L1, {(taxonomy_df['level']=='L2').sum()} L2)")
taxonomy_df.head(10)

Write the parsed taxonomy into the `taxonomy` sheet, preserving `README`/`modules`/`fields` untouched.

In [ ]:
with pd.ExcelWriter(DEFINITION_XLSX, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    taxonomy_df.to_excel(writer, sheet_name="taxonomy", index=False)
print(f"Taxonomy sheet written: {len(taxonomy_df)} rows -> {DEFINITION_XLSX.name}")

### 1a. Disable `root_cause`, migrate its justification/evidence fields into `contact_reason`

`contact_reason` and `root_cause` are merged into a single hierarchical module (L1=Contact Reason, L2=Root Cause) rather than kept as two independent modules -- this mirrors the proven pattern in `notebooks/04_leapspace_classify_with_gold_set.ipynb` for the same kind of taxonomy, and avoids a real technical gap (contact_reason alone has no natural second level, and the pipeline's L1-code auto-repair logic needs at least one child per category).

`root_cause` is **disabled, not deleted** (`enabled=FALSE`), per the modules sheet's own documented convention. This cell is idempotent -- safe to re-run.

In [ ]:
sheets = pd.read_excel(DEFINITION_XLSX, sheet_name=None, engine="openpyxl")
modules_df = sheets["modules"]
fields_df  = sheets["fields"]

# -- Disable root_cause, update contact_reason description/notes ------------
# .astype(object) first: pandas 3.x raises TypeError assigning a bool into an
# already-float64 "enabled" column (which is what Excel round-trips TRUE/FALSE
# checkboxes as) -- this keeps the cell safe to re-run.
modules_df["enabled"] = modules_df["enabled"].astype(object)
modules_df.loc[modules_df["module_name"] == "root_cause", "enabled"] = False
modules_df.loc[modules_df["module_name"] == "root_cause", "notes"] = (
    "DISABLED 2026-07-02 -- merged into contact_reason module "
    "(fields renamed contact_reason_justification / contact_reason_evidence / "
    "contact_reason_evidence_type). Preserved per modules-sheet convention: "
    "'Set enabled=FALSE to exclude a module from the pipeline without deleting it.'"
)
modules_df.loc[modules_df["module_name"] == "contact_reason", "description"] = (
    "Hierarchical classification of the customer's Contact Reason (L1) AND Root "
    "Cause (L2) for Evolve support tickets. L1 = Contact Reason category (11 "
    "groups); L2 = the specific Root Cause valid within that Contact Reason -- "
    "the same root-cause label (e.g. 'Other', 'Not enough info') legitimately "
    "repeats under multiple different L1 parents with distinct descriptions. "
    "Assign the single best-fitting LEAF code (e.g. CR4.2)."
)
modules_df.loc[modules_df["module_name"] == "contact_reason", "notes"] = (
    "Hierarchical: L1=Contact Reason, L2=Root Cause. Merged root_cause module "
    "fields folded in with contact_reason_ prefix (see fields sheet)."
)

# -- Add migrated evidence/justification fields, idempotently ---------------
_new_field_names = {"contact_reason_justification", "contact_reason_evidence", "contact_reason_evidence_type"}
_already_present = set(fields_df.loc[fields_df["module_name"] == "contact_reason", "field_name"]) & _new_field_names
if _already_present:
    print(f"Skipping already-present fields: {_already_present}")

new_fields = pd.DataFrame([
    {"module_name": "contact_reason", "field_name": "contact_reason_justification",
     "field_type": "str",
     "description": "1-2 sentence reasoning chain linking evidence to the chosen code.",
     "prompt_instruction": (
         "Explain WHY you chose this root cause (L2 leaf) code, linking the "
         "evidence from the transcript to the taxonomy definition (1-3 sentences)."),
     "in_output": True, "explode_list": False, "validate_enum": False,
     "notes": "Migrated from root_cause_justification."},
    {"module_name": "contact_reason", "field_name": "contact_reason_evidence",
     "field_type": "str",
     "description": "Most compelling piece of evidence from the transcript.",
     "prompt_instruction": (
         "Copy verbatim the single most relevant sentence or phrase from the "
         "transcript text above that most strongly supports the contact reason "
         "/ root cause assignment. Do NOT paraphrase, infer, or add words that "
         "are not in the transcript. Do NOT invent product names, system "
         "names, or details not explicitly stated. If no direct evidence "
         "exists, return an empty string."),
     "in_output": True, "explode_list": False, "validate_enum": False,
     "notes": "Migrated from root_cause_evidence. Auto-validated (*_grounded column)."},
    {"module_name": "contact_reason", "field_name": "contact_reason_evidence_type",
     "field_type": "enum:customer_symptom,agent_action,system_message",
     "description": "Source of the evidence.",
     "prompt_instruction": (
         "Classify where the evidence comes from: customer_symptom (customer "
         "described the problem), agent_action (agent discovered it during "
         "the interaction), system_message (a system or platform error message)."),
     "in_output": True, "explode_list": False, "validate_enum": True,
     "notes": "Migrated from root_cause_evidence_type."},
])
new_fields = new_fields[~new_fields["field_name"].isin(_already_present)]
fields_df = pd.concat([fields_df, new_fields], ignore_index=True)

sheets["modules"] = modules_df
sheets["fields"]  = fields_df
with pd.ExcelWriter(DEFINITION_XLSX, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    for name, df in sheets.items():
        df.to_excel(writer, sheet_name=name, index=False)

print("modules: root_cause.enabled -> False, contact_reason description/notes updated")
print(f"fields: {len(new_fields)} new field(s) added to contact_reason")

Sanity check: reload from disk and confirm nothing was lost.

In [ ]:
_check = pd.read_excel(DEFINITION_XLSX, sheet_name=None, engine="openpyxl")
print("modules sheet:")
print(_check["modules"][["module_name", "enabled", "taxonomy_prefix"]].to_string(index=False))
print(f"\ncontact_reason fields ({len(_check['fields'][_check['fields']['module_name']=='contact_reason'])}):",
      _check["fields"][_check["fields"]["module_name"] == "contact_reason"]["field_name"].tolist())
print(f"root_cause fields (should still be 7, unchanged):",
      _check["fields"][_check["fields"]["module_name"] == "root_cause"]["field_name"].tolist())
print(f"taxonomy sheet: {len(_check['taxonomy'])} rows, all contact_reason: "
      f"{(_check['taxonomy']['module_name']=='contact_reason').all()}")

assert (_check["modules"].loc[_check["modules"]["module_name"] == "root_cause", "enabled"] == False).all()
assert len(_check["fields"][_check["fields"]["module_name"] == "contact_reason"]) == 8
assert len(_check["fields"][_check["fields"]["module_name"] == "root_cause"]) == 7
assert len(_check["taxonomy"]) == 65
print("\nAll assertions passed.")

## 2. Run `rca build` to regenerate the taxonomy JSON and classifier module

**Note:** `rca build build` regenerates `generated/pipeline_config.json` and `generated/modules/contact_reason.py` using only the modules defined in this workbook. If you've been working with a different product's definition file in this repo, re-run that product's build before returning to it -- this is pre-existing, shared `generated/` behaviour, not something introduced here. Taxonomy JSONs are written to a product-specific `--taxonomy-dir` (`data/evolve_taxonomies/`) instead of the shared `generated/taxonomies/`, to avoid clobbering other products' taxonomies.

`--force contact_reason` only: `root_cause` is now disabled and would be skipped by the build regardless; only `contact_reason`'s definition actually changed.

In [ ]:
import subprocess, sys

TAXONOMY_DIR.mkdir(parents=True, exist_ok=True)

result = subprocess.run(
    [sys.executable, "-m", "rca_pipeline.cli", "build",
     "--input", str(DEFINITION_XLSX),
     "--taxonomy-dir", str(TAXONOMY_DIR),
     "--force", "contact_reason"],
    capture_output=True, text=True, encoding="utf-8", errors="replace", cwd=str(BASE),
    env=_SUBPROCESS_ENV,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError("rca build failed")
print("Build complete.")

In [ ]:
import json

with open(TAXONOMY_DIR / "contact_reason.json", encoding="utf-8") as f:
    tax_json = json.load(f)
n_l1 = len(tax_json)
n_l2 = sum(len(v.get("children", {})) for v in tax_json.values())
print(f"contact_reason.json: {n_l1} L1 groups, {n_l2} L2 leaves")
for code_, node in tax_json.items():
    print(f"  [{code_}] {node['label']}  ({len(node.get('children', {}))} children)")

assert n_l1 == 11 and n_l2 == 54
_module_py = BASE / "generated" / "modules" / "contact_reason.py"
assert _module_py.exists(), "contact_reason.py was not generated"
print(f"\ngenerated/modules/contact_reason.py regenerated at {_module_py.stat().st_mtime}")

## 3. Build the label -> code lookup (for `FeedbackStore` and evaluation)

In [ ]:
def build_label_to_code_lookup(taxonomy_json: dict) -> dict[str, dict[str, str]]:
    """taxonomy_json: loaded contact_reason.json -- {l1_code: {label, children: {leaf_code: {label,...}}}}.
    Returns hierarchy[cr_label][rc_label] -> leaf_code (mirrors data/leapspace_code_map.json)."""
    hierarchy = {}
    for l1_node in taxonomy_json.values():
        hierarchy[l1_node["label"]] = {
            leaf["label"]: code_ for code_, leaf in l1_node.get("children", {}).items()
        }
    return hierarchy


hierarchy = build_label_to_code_lookup(tax_json)
with open(CODE_MAP_PATH, "w", encoding="utf-8") as f:
    json.dump({"hierarchy": hierarchy}, f, indent=2, ensure_ascii=False)
print(f"Code map: {sum(len(v) for v in hierarchy.values())} CR->RC leaf codes -> {CODE_MAP_PATH.name}")

## 4. Load the gold set and resolve labels to codes

In [ ]:
gold = pd.read_excel(GOLD_SET_PATH, sheet_name="evolve")
gold["Contact Reason Final Decision"] = gold["Contact Reason Final Decision"].astype(str).str.strip()
gold["Root Cause Final Decision"]     = gold["Root Cause Final Decision"].astype(str).str.strip()

# Ticket Number must be unique -- a duplicate uid breaks the k-fold prediction lookup
# later (pandas .loc on a non-unique index returns a DataFrame, not a Series). The
# gold set has 2 genuine duplicate tickets, both with identical CR/RC labels on
# both copies, so dropping the second copy loses no information.
_n_before = len(gold)
gold = gold.drop_duplicates(subset=["Ticket Number"], keep="first").reset_index(drop=True)
if len(gold) < _n_before:
    print(f"Dropped {_n_before - len(gold)} duplicate Ticket Number row(s) (identical labels on both copies)")

print(f"Gold set: {len(gold)} tickets")
print(f"Columns: {gold.columns.tolist()}")
print("\nContact Reason distribution:")
print(gold["Contact Reason Final Decision"].value_counts().to_string())
print("\nRoot Cause distribution:")
print(gold["Root Cause Final Decision"].value_counts().to_string())

### Overlay the LLM-checked corrections

`CHECKED_SET_PATH` reviews the 170 tickets whose Root-Cause prediction disagreed with gold in a prior run. Its `Final cr` / `Final rc` columns hold the reviewer's verdicts (filled for 51 rows) and `Reason` explains each. Corrections are merged onto the gold set by `Ticket Number`; rows left blank keep their original gold label (24 CR + 32 RC labels change). A few loosely-written Contact Reason names (`Other`, `Other Access`) are canonicalized to `Other access related`, mirroring the existing `_ALIAS_CR` mechanism below. The `To check = X` flag is an internal annotation and is not used to filter (all reviewed rows carry a verdict + reason). Each `Reason` is attached to its ticket and used as the few-shot `reason` field in Sections 5 and 9. Any (CR, RC) pair that still doesn't match the taxonomy is handled by the existing resolution step below (excluded, not silently dropped).

In [ ]:
# --- Overlay the LLM-checked label corrections ----------------------------
checked = pd.read_excel(CHECKED_SET_PATH, sheet_name="in")
checked["uid"] = checked["uid"].astype(str).str.strip()

# The reviewer wrote a few Contact Reason names in loose form; map them to the
# canonical taxonomy labels (mirrors the _ALIAS_CR mechanism in the next cell).
_FINAL_CR_CANON = {"other": "Other access related", "other access": "Other access related"}
def _canon_cr(x):
    return _FINAL_CR_CANON.get(str(x).strip().lower(), x) if pd.notna(x) else x
checked["Final cr"] = checked["Final cr"].map(_canon_cr)

# Defense-in-depth: every reviewed uid must be a real Ticket Number in the gold
# set (the join key) -- abort loudly rather than silently miss corrections.
_gold_tn = set(gold["Ticket Number"].astype(str).str.strip())
_missing = sorted(set(checked["uid"]) - _gold_tn)
assert not _missing, f"errors-file uids not found in gold Ticket Number: {_missing}"
print(f"uid validation passed: all {checked['uid'].nunique()} reviewed uids are in the gold set")

_final_cr = dict(zip(checked["uid"], checked["Final cr"]))
_final_rc = dict(zip(checked["uid"], checked["Final rc"]))
_reason   = dict(zip(checked["uid"], checked["Reason"]))

# Overlay each label where the reviewer filled a verdict; attach the reason so
# Sections 5 and 9 can use it as the few-shot `reason` (highest-weight signal).
# Use pd.notna, NOT a bare truthiness check: a blank Excel cell reads back as
# NaN (float), and bool(NaN) is True -- a bare `if _fc` would overwrite good
# gold labels with the string "nan" (and this differs across pandas versions).
def _present(v):
    return pd.notna(v) and str(v).strip() != ""

gold["correction_reason"] = ""
_n_cr = _n_rc = _n_reason = 0
for _i, _row in gold.iterrows():
    _u = str(_row["Ticket Number"]).strip()
    _fc, _fr, _rs = _final_cr.get(_u), _final_rc.get(_u), _reason.get(_u)
    if _present(_fc) and str(_fc).strip() != _row["Contact Reason Final Decision"]:
        gold.at[_i, "Contact Reason Final Decision"] = str(_fc).strip(); _n_cr += 1
    if _present(_fr) and str(_fr).strip() != _row["Root Cause Final Decision"]:
        gold.at[_i, "Root Cause Final Decision"] = str(_fr).strip(); _n_rc += 1
    if _present(_rs):
        gold.at[_i, "correction_reason"] = str(_rs).strip(); _n_reason += 1

print(f"Applied LLM-checked corrections: {_n_cr} CR labels, {_n_rc} RC labels changed")
print(f"Attached reviewer reasons to {_n_reason} gold rows (used as few-shot 'reason').")
print("\nCorrected Contact Reason distribution:")
print(gold["Contact Reason Final Decision"].value_counts().to_string())
print("\nCorrected Root Cause distribution:")
print(gold["Root Cause Final Decision"].value_counts().to_string())

**Label normalization.** The gold set's `Final Decision` columns don't always match the `Definition` sheet's labels verbatim -- e.g. `Access/QR code` vs `Access / QR code`, `Lack of User Guidance` vs `Lack of user guidance`, or `Update email or Account update` vs `Update email / Account update`. Root Cause labels are also **not globally unique** (the same label like "Other" or "Not enough info" appears under many different Contact Reason parents with different meanings), so matching must resolve Contact Reason first, then look up Root Cause scoped to that parent only.

In [ ]:
import re

_ALIAS_CR = {
    "other - access related (please provide suggestion)": "other access related",
    "other - not access related": "other not access related",
}

def _norm_label(s: str) -> str:
    s = re.sub(r"\s+", " ", str(s).strip().lower())
    s = re.sub(r"\s*/\s*", "/", s)      # "a / b" and "a/b" -> "a/b"
    s = re.sub(r"\s+or\s+", "/", s)     # "a or b" -> "a/b"
    return s

def resolve_cr_rc_to_code(cr_raw: str, rc_raw: str, hierarchy: dict) -> tuple:
    """Returns (leaf_code_or_None, matched_cr_label, matched_rc_label).
    RC matching is scoped to the resolved CR's children only -- RC labels are not globally unique."""
    cr_norm = _norm_label(cr_raw)
    cr_norm = _ALIAS_CR.get(cr_norm, cr_norm)
    cr_lookup = {_norm_label(k): k for k in hierarchy}
    matched_cr = cr_lookup.get(cr_norm)
    if matched_cr is None:
        return None, "", ""

    rc_candidates = hierarchy[matched_cr]
    rc_norm = _norm_label(rc_raw)
    rc_lookup = {_norm_label(k): k for k in rc_candidates}
    matched_rc = rc_lookup.get(rc_norm)
    if matched_rc is None:
        return None, matched_cr, ""

    return rc_candidates[matched_rc], matched_cr, matched_rc


resolved = gold.apply(
    lambda row: resolve_cr_rc_to_code(
        row["Contact Reason Final Decision"], row["Root Cause Final Decision"], hierarchy
    ), axis=1, result_type="expand"
)
resolved.columns = ["leaf_code", "matched_cr", "matched_rc"]
gold = pd.concat([gold, resolved], axis=1)

n_resolved   = gold["leaf_code"].notna().sum()
n_unresolved = gold["leaf_code"].isna().sum()
print(f"Resolved: {n_resolved}/{len(gold)}  |  Unresolved: {n_unresolved}/{len(gold)}")

if n_unresolved:
    print("\nUnresolved rows (genuine gold-set labeling inconsistencies -- excluded below, not silently dropped):")
    print(gold[gold["leaf_code"].isna()][
        ["Ticket Number", "Contact Reason Final Decision", "Root Cause Final Decision"]
    ].to_string(index=False))

In [ ]:
gold_usable = gold[gold["leaf_code"].notna()].reset_index(drop=True)
print(f"Usable gold rows for few-shot/evaluation: {len(gold_usable)} / {len(gold)}")

## 5. Write the gold set to `FeedbackStore` JSONL

Each ticket becomes a feedback entry where `original == corrected` (the human label is treated as an authoritative correction), matching the `04_leapspace_classify_with_gold_set.ipynb` pattern. **Note:** unlike LeapSpace's gold set (which has an `Explanations` column feeding the `reason` field, the highest-weighted signal in the `smart` strategy), the Evolve gold set has no free-text rationale column, so `reason` is left blank. This slightly reduces the `smart` strategy's `has_reason` scoring term but doesn't break it -- content-overlap and recency still differentiate examples.

No ID matching is needed between the gold set and the raw dataset -- the gold set's text column (`Reminding of ticket content`) and the raw dataset's (`Conversation`) are different things with no ID/text overlap. The gold set is used purely as few-shot prompt context, never joined by ID to the raw 20,070-row 2026 dataset.

In [ ]:
import json
from datetime import datetime, timezone

FEEDBACK_DIR.mkdir(parents=True, exist_ok=True)

def write_feedback_jsonl(df: pd.DataFrame, out_path: Path) -> list[dict]:
    entries = []
    for _, row in df.iterrows():
        entries.append({
            "uid":       str(row["Ticket Number"]),
            "text":      str(row["Reminding of ticket content"])[:1500],
            "original":  {"contact_reason_primary_code": row["leaf_code"]},
            "corrected": {"contact_reason_primary_code": row["leaf_code"]},
            "reason":    ("" if pd.isna(row.get("correction_reason")) else str(row["correction_reason"])),
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "module":    "contact_reason",
        })
    with open(out_path, "w", encoding="utf-8") as f:
        for e in entries:
            f.write(json.dumps(e, ensure_ascii=False) + "\n")
    print(f"  {out_path.name}: {len(entries)} entries")
    return entries


cr_entries = write_feedback_jsonl(gold_usable, FEEDBACK_DIR / "contact_reason_corrections.jsonl")

_old_rc = FEEDBACK_DIR / "root_cause_corrections.jsonl"
if _old_rc.exists():
    _old_rc.unlink()
    print(f"  Removed stale: {_old_rc.name}")

print("\nFeedback JSONL ready.")

Sanity check: verify a sample entry decodes back to a sensible (Contact Reason, Root Cause) pair.

In [ ]:
print(json.dumps(cr_entries[0], indent=2, ensure_ascii=False))
sample_code = cr_entries[0]["corrected"]["contact_reason_primary_code"]
for cr, rc_map in hierarchy.items():
    for rc, code_ in rc_map.items():
        if code_ == sample_code:
            print(f"\n  {sample_code}  =>  Contact Reason: '{cr}'  /  Root Cause: '{rc}'")

## 6. Preview the raw dataset, filter to 2026, and sample

In [ ]:
full_df = pd.read_excel(RAW_DATA_PATH, sheet_name="Sheet1")
print(f"Full dataset: {len(full_df):,} rows")
print(f"Columns: {full_df.columns.tolist()}")
full_df.head(3)

In [ ]:
full_df["Chat_Start_Date"] = pd.to_datetime(full_df["Chat_Start_Date"], errors="coerce")
df_2026 = full_df[full_df["Chat_Start_Date"].dt.year == SCOPE_YEAR].reset_index(drop=True)
print(f"Filtered to {SCOPE_YEAR}: {len(full_df):,} -> {len(df_2026):,} rows")

In [ ]:
if SAMPLE_N is not None:
    work_df = df_2026.sample(n=min(SAMPLE_N, len(df_2026)), random_state=SAMPLE_SEED).reset_index(drop=True)
    print(f"Sampled {len(work_df):,} / {len(df_2026):,} {SCOPE_YEAR} rows (SAMPLE_N={SAMPLE_N}, seed={SAMPLE_SEED})")
else:
    work_df = df_2026
    print(f"SAMPLE_N=None -> using FULL {SCOPE_YEAR} slice: {len(work_df):,} rows")
    print("WARNING: this will make real LLM API calls against all rows. "
          "Estimated cost at this notebook's own measured ~$0.007/ticket: "
          f"~${len(work_df) * 0.007:,.0f}. Estimated time at ~5 rec/s: "
          f"~{len(work_df) / 5 / 3600:.1f} hours.")

_tmp_input = BASE / "runs" / "_evolve_sample_tmp.xlsx"
_tmp_input.parent.mkdir(parents=True, exist_ok=True)
work_df.to_excel(_tmp_input, index=False)

## 7. Run the pipeline via `rca run`

`--feedback-strategy smart` uses `FeedbackStore`'s two-stage selection: pick a candidate pool weighted by recency/content/has-reason, then rerank per-batch by token-overlap with that batch's transcripts. `--track-costs` prints a running cost-per-ticket estimate.

In [ ]:
import subprocess, sys, threading

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, '-m', 'rca_pipeline.cli', 'run',
    "--input",                 str(_tmp_input),
    "--output",                str(OUTPUT_PATH),
    "--id-col",                ID_COL,
    "--text-col",              TEXT_COL,
    "--modules",                "contact_reason",
    "--feedback-dir",          str(FEEDBACK_DIR),
    "--feedback-strategy",     FEEDBACK_STRATEGY,
    "--feedback-max-examples", str(FEEDBACK_MAX),
    "--batch-size",            str(BATCH_SIZE),
    "--concurrency",           str(CONCURRENCY),
    "--track-costs",
    "--taxonomy-dir",          str(TAXONOMY_DIR),
]

print("Running:", " ".join(str(c) for c in cmd))
print()

_err_lines: list[str] = []
def _capture_err(proc):
    for line in iter(proc.stderr.readline, ""):
        _err_lines.append(line)

proc = subprocess.Popen(
    cmd, text=True, encoding="utf-8", errors="replace",
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, bufsize=1,
    env=_SUBPROCESS_ENV,
)
_t = threading.Thread(target=_capture_err, args=(proc,), daemon=True)
_t.start()
for _line in iter(proc.stdout.readline, ""):
    print(_line, end="", flush=True)
proc.wait()
_t.join(timeout=5)

if proc.returncode == 0:
    print(f"\n-> Pipeline complete. Output: {OUTPUT_PATH}")
else:
    print(f"\n-> Pipeline exited with code {proc.returncode}")
    if _err_lines:
        print("\n--- Errors (last 60 lines) ---")
        print("".join(_err_lines[-60:]))
    raise RuntimeError(f"rca run failed with exit code {proc.returncode}")

## 8. Inspect results

In [ ]:
results = pd.read_excel(OUTPUT_PATH)
print(f"Output rows: {len(results):,}")
print(f"Output columns: {results.columns.tolist()}")
results.head(5)

In [ ]:
print("Predicted Contact Reason (L1) distribution:")
print(results["contact_reason_class_l1_label"].value_counts().to_string())
print("\nPredicted Root Cause (L2) distribution:")
print(results["contact_reason_primary_label"].value_counts().to_string())

**Verification checks** -- codes match the taxonomy, evidence grounding is wired up, and no (L1, L2) pair is nonsensical (i.e. a Root Cause paired with a Contact Reason it doesn't belong to).

In [ ]:
import re as _re

_pattern_ok = results["contact_reason_primary_code"].astype(str).str.match(r"^CR([1-9]|1[01])\.\d+$").all()
print("All codes match CR{1-11}.{n} pattern:", _pattern_ok)

def _check_pair(row):
    return row["contact_reason_primary_label"] in hierarchy.get(row["contact_reason_class_l1_label"], {})

_valid_pairs = results.apply(_check_pair, axis=1)
print(f"Valid (L1,L2) pairs: {_valid_pairs.sum()}/{len(results)}")
if not _valid_pairs.all():
    print(results.loc[~_valid_pairs, ["contact_reason_class_l1_label", "contact_reason_primary_label"]])

print("contact_reason_evidence_grounded present:", "contact_reason_evidence_grounded" in results.columns)
if "contact_reason_evidence_grounded" in results.columns:
    print("  value counts:", results["contact_reason_evidence_grounded"].value_counts(dropna=False).to_dict())

## 9. Gold-set evaluation -- 5-fold cross-validation

This is the primary accuracy check: each fold uses ~227 gold tickets as few-shot examples and evaluates on the ~57 held out, so every gold ticket eventually gets scored as an unseen prediction. Stratified on Contact Reason so each fold sees a representative class mix (all 10 real Contact Reason classes have >=5 instances, so none need to fall into a "rare" bucket).

This loop is embedded directly in the notebook (rather than calling a separate script like `scripts/run_kfold_eval.py`) because Evolve's label resolution (`resolve_cr_rc_to_code`) is nontrivial enough -- multi-tier normalization + parent-scoped matching -- that keeping one copy of it here avoids two implementations drifting apart.

In [ ]:
import shutil, time
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import StratifiedKFold

N_FOLDS      = 5
RANDOM_STATE = 42
TMP_KFOLD_DIR      = BASE / "runs" / "tmp_kfold_evolve"
KFOLD_RESULTS_PATH = BASE / "runs" / "output" / "evolve_kfold_eval_results.xlsx"


def _write_fold_feedback(fold_train: pd.DataFrame, out_path: Path) -> None:
    entries = []
    for _, row in fold_train.iterrows():
        entries.append({
            "uid": str(row["Ticket Number"]),
            "text": str(row["Reminding of ticket content"])[:1500],
            "original": {"contact_reason_primary_code": row["leaf_code"]},
            "corrected": {"contact_reason_primary_code": row["leaf_code"]},
            "reason": ("" if pd.isna(row.get("correction_reason")) else str(row["correction_reason"])),
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "module": "contact_reason",
        })
    with open(out_path, "w", encoding="utf-8") as f:
        for e in entries:
            f.write(json.dumps(e, ensure_ascii=False) + "\n")


def _run_fold(fold_idx: int, test_df: pd.DataFrame, feedback_dir: Path):
    fold_dir = TMP_KFOLD_DIR / f"fold_{fold_idx}"
    fold_dir.mkdir(parents=True, exist_ok=True)
    in_csv, out_xlsx = fold_dir / "fold_in.csv", fold_dir / "fold_out.xlsx"
    test_df[["uid", "conversation"]].to_csv(in_csv, index=False)
    cmd = [
        sys.executable, '-m', 'rca_pipeline.cli', 'run',
        "--input", str(in_csv), "--output", str(out_xlsx),
        "--text-col", "conversation", "--id-col", "uid",
        "--modules", "contact_reason",
        "--taxonomy-dir", str(TAXONOMY_DIR),
        "--feedback-dir", str(feedback_dir),
        "--feedback-strategy", FEEDBACK_STRATEGY,
        "--feedback-max-examples", str(FEEDBACK_MAX),
        "--batch-size", str(BATCH_SIZE), "--concurrency", str(CONCURRENCY),
    ]
    proc = subprocess.run(cmd, text=True, encoding="utf-8", errors="replace", capture_output=True, env=_SUBPROCESS_ENV)
    if proc.returncode != 0 or not out_xlsx.exists():
        print(f"  [fold {fold_idx + 1}] failed: {proc.stderr[-2000:]}")
        return None
    return pd.read_excel(out_xlsx)


TMP_KFOLD_DIR.mkdir(parents=True, exist_ok=True)
cr_counts = gold_usable["Contact Reason Final Decision"].value_counts()
stratify = gold_usable["Contact Reason Final Decision"].where(
    cr_counts.reindex(gold_usable["Contact Reason Final Decision"]).values >= N_FOLDS, "rare"
)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
all_records = []
t0 = time.time()

for fold_idx, (train_idx, test_idx) in enumerate(skf.split(gold_usable, stratify)):
    fold_train = gold_usable.iloc[train_idx].copy()
    fold_test  = gold_usable.iloc[test_idx].copy()
    print(f"Fold {fold_idx + 1}/{N_FOLDS}  ({len(fold_train)} few-shot | {len(fold_test)} test)")

    fb_dir = TMP_KFOLD_DIR / f"feedback_{fold_idx}"
    fb_dir.mkdir(parents=True, exist_ok=True)
    _write_fold_feedback(fold_train, fb_dir / "contact_reason_corrections.jsonl")

    test_input = fold_test.rename(
        columns={"Ticket Number": "uid", "Reminding of ticket content": "conversation"}
    )[["uid", "conversation"]].copy()
    test_input["uid"] = test_input["uid"].astype(str)

    t_fold = time.time()
    pred_df = _run_fold(fold_idx, test_input, fb_dir)
    fold_s = time.time() - t_fold
    if pred_df is None:
        print(f"  Fold {fold_idx + 1} failed -- skipping")
        continue

    pred_df["uid"] = pred_df["uid"].astype(str).str.strip()
    indexed = pred_df.set_index("uid")
    fold_rows = []
    for _, row in fold_test.iterrows():
        uid = str(row["Ticket Number"])
        if uid not in indexed.index:
            continue
        pred_row = indexed.loc[uid]
        cr_pred = str(pred_row.get("contact_reason_class_l1_label", "")).strip()
        rc_pred = str(pred_row.get("contact_reason_primary_label", "")).strip()
        fold_rows.append({
            "uid": uid, "gold_cr": row["matched_cr"], "gold_rc": row["matched_rc"],
            "pred_cr": cr_pred, "pred_rc": rc_pred,
        })
    all_records.extend(fold_rows)

    valid_cr = [r for r in fold_rows if r["pred_cr"]]
    cr_acc = accuracy_score([r["gold_cr"] for r in valid_cr], [r["pred_cr"] for r in valid_cr]) * 100 if valid_cr else float("nan")
    print(f"  Fold {fold_idx + 1} done in {fold_s:.0f}s  |  CR acc: {cr_acc:.1f}%  ({len(fold_rows)}/{len(fold_test)} matched)")

df_kfold = pd.DataFrame(all_records)
print(f"\nAggregate: {len(df_kfold)} predictions across {N_FOLDS} folds  ({(time.time()-t0)/60:.1f} min total)")

In [ ]:
for label, gold_col, pred_col in [("Contact Reason (L1)", "gold_cr", "pred_cr"), ("Root Cause (L2)", "gold_rc", "pred_rc")]:
    valid = df_kfold[df_kfold[pred_col].str.strip() != ""]
    if valid.empty:
        print(f"\n{label}: no predictions found")
        continue
    acc = accuracy_score(valid[gold_col], valid[pred_col]) * 100
    print(f"\n{label}  (n={len(valid)})  Accuracy: {acc:.1f}%")
    print(classification_report(valid[gold_col], valid[pred_col], zero_division=0))

df_kfold.to_excel(KFOLD_RESULTS_PATH, index=False)
shutil.rmtree(TMP_KFOLD_DIR, ignore_errors=True)
print(f"\nResults saved: {KFOLD_RESULTS_PATH}")

## 10. Record-level error analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

errors_cr = df_kfold[df_kfold["gold_cr"] != df_kfold["pred_cr"]]
print(f"Contact Reason errors: {len(errors_cr)}/{len(df_kfold)}")
if len(errors_cr):
    pairs = errors_cr.groupby(["gold_cr", "pred_cr"]).size().sort_values(ascending=False).head(15)
    print(pairs.to_string())

errors_rc = df_kfold[(df_kfold["gold_rc"] != df_kfold["pred_rc"]) & (df_kfold["pred_rc"].str.strip() != "")]
print(f"\nRoot Cause errors: {len(errors_rc)}/{len(df_kfold)}")
if len(errors_rc):
    pairs_rc = errors_rc.groupby(["gold_cr", "gold_rc", "pred_rc"]).size().sort_values(ascending=False).head(15)
    print(pairs_rc.to_string())

In [ ]:
# errors_rc.to_csv("runs\output\evolve_Root_Cause_errors.csv", index=False) 

In [ ]:
top_classes = df_kfold["gold_cr"].value_counts().head(11).index.tolist()
mask = df_kfold["gold_cr"].isin(top_classes)
cm = confusion_matrix(df_kfold.loc[mask, "gold_cr"], df_kfold.loc[mask, "pred_cr"], labels=top_classes)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=top_classes, yticklabels=top_classes, cmap="Blues", ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("Gold (True)")
ax.set_title("Contact Reason confusion matrix (5-fold aggregate)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 11. Adding more corrections

After reviewing output, add corrections for misclassified tickets directly via `FeedbackStore` -- these are picked up automatically on the next run.

```python
from rca_pipeline.enhancements.feedback import FeedbackStore

store = FeedbackStore(FEEDBACK_DIR)
store.add_correction(
    module="contact_reason",
    uid="some_ticket_id",
    text="transcript text here...",
    original={"contact_reason_primary_code": "CR1.2"},
    corrected={"contact_reason_primary_code": "CR3.1"},
    reason="The customer's main concern was billing, not access itself",
)
```

The `reason` field has the highest weight in the `smart` strategy's scoring -- always fill it in.

---

**Scope note:** this notebook targets only the 2026 slice of `Evolve_Collapsed.xlsx` (20,070 of 126,504 rows) and only the `contact_reason` module. `resolution`, `escalation`, `troubleshooting`, and `metadata` remain enabled in `evolve_pipeline_definition.xlsx` but are out of scope here -- there's no gold-set ground truth to build their taxonomies from or validate their output against. The full `SAMPLE_N = None` run over all 20,070 2026 rows is a deliberate, costlier action left for you to trigger once satisfied with sample accuracy.